# Comparative Legal Clause Classification

This notebook builds a reproducible NLP coursework pipeline for legal clause classification using LEDGAR as the main dataset. It compares dummy baselines, sparse classical NLP models, a fine-tuned transformer classifier, optional Qwen2.5-Instruct prompting, and a small human-in-the-loop review prototype.

The task is clause type classification: given a contract clause or provision, predict the clause category. The notebook does not make legal risk predictions and does not provide legal advice.

## 1. Install, Imports, and Configuration

The configuration flags below control expensive sections. Classical models should run on CPU. Transformer and Qwen sections use GPU automatically when available and skip gracefully if the runtime is unsuitable.

In [ ]:
from pathlib import Path
import difflib
import importlib.util
import json
import os
import random
import re
import shutil
import subprocess
import sys
import warnings
from collections import Counter
from typing import Any


def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    """Install a missing notebook dependency."""
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])


for import_name, pip_name in [
    ("pandas", "pandas"),
    ("datasets", "datasets"),
    ("huggingface_hub", "huggingface_hub"),
    ("sklearn", "scikit-learn"),
    ("joblib", "joblib"),
    ("matplotlib", "matplotlib"),
]:
    ensure_package(import_name, pip_name)

from datasets import load_dataset
from huggingface_hub import hf_hub_download, snapshot_download
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

SEED = 42
DATASET_NAME = "LEDGAR"
TOP_K_LABELS = 20
MAX_FEATURES_LIST = [10000, 30000]
NGRAM_RANGES = [(1, 1), (1, 2)]
RUN_CLASSICAL_MODELS = True
RUN_TRANSFORMER = True
RUN_QWEN_BASELINE = True
RUN_AGENTIC_EXTENSION = True
RUN_NAIVE_BAYES = True
TRANSFORMER_MODEL_NAME = "distilbert-base-uncased"
OPTIONAL_LEGAL_MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
QWEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_TRANSFORMER_LENGTH = 256
QWEN_EVAL_SAMPLE_SIZE = 200
QWEN_FEW_SHOT_EXAMPLES_PER_CLASS = 1

DOWNLOAD_LEDGAR_IF_MISSING = True
DOWNLOAD_CUAD_IF_MISSING = True
USE_HF_CACHE = True
FORCE_REDOWNLOAD = False

PROJECT_ROOT = Path(".").resolve()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
LEDGAR_RAW_DIR = RAW_DATA_DIR / "ledgar"
LEGACY_LEDGAR_RAW_DIR = RAW_DATA_DIR / "lexglue_ledgar"
CUAD_RAW_DIR = RAW_DATA_DIR / "cuad"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

for path in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    LEDGAR_RAW_DIR,
    CUAD_RAW_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is unavailable. Transformer/Qwen sections will skip or reduce work gracefully.")

## 2. Dataset Download and Raw Setup

LEDGAR is downloaded with Hugging Face Datasets because it is already structured with train, validation, and test splits for legal clause classification.

CUAD is downloaded separately because it is organised as a contract review, question-answering, and span-extraction dataset rather than a direct classification dataset. CUAD is not merged with LEDGAR. It can be adapted later by extracting annotated answer spans and assigning each span the corresponding CUAD category.

In [ ]:
def json_safe(value: Any) -> Any:
    """Convert numpy/pandas objects into JSON-safe values."""
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    try:
        if pd.isna(value) and not isinstance(value, str):
            return None
    except Exception:
        pass
    return value


def write_json(path: Path | str, payload: Any) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_safe(payload), ensure_ascii=False, indent=2), encoding="utf-8")
    return path


def save_jsonl(df: pd.DataFrame, path: Path | str) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_json(path, orient="records", lines=True, force_ascii=False)
    return path


def load_jsonl(path: Path | str) -> pd.DataFrame:
    return pd.read_json(path, lines=True)


def ledgar_raw_paths(raw_dir: Path = LEDGAR_RAW_DIR) -> dict[str, Path]:
    return {split: raw_dir / f"ledgar_{split}.jsonl" for split in ["train", "validation", "test"]}


def legacy_ledgar_raw_paths(raw_dir: Path = LEGACY_LEDGAR_RAW_DIR) -> dict[str, Path]:
    return {split: raw_dir / f"ledgar_{split}.jsonl" for split in ["train", "validation", "test"]}


def load_local_ledgar_jsonl(raw_dir: Path) -> dict[str, pd.DataFrame] | None:
    paths = ledgar_raw_paths(raw_dir) if raw_dir.name == "ledgar" else legacy_ledgar_raw_paths(raw_dir)
    if all(path.exists() for path in paths.values()):
        return {split: load_jsonl(path) for split, path in paths.items()}
    return None


def load_or_download_ledgar() -> dict[str, pd.DataFrame] | None:
    """
    Load LEDGAR from local JSONL files if available.
    Otherwise download from Hugging Face using load_dataset("coastalcph/lex_glue", "ledgar").
    """
    expected_paths = ledgar_raw_paths()
    if not FORCE_REDOWNLOAD and all(path.exists() for path in expected_paths.values()):
        print("Loading LEDGAR from data/raw/ledgar JSONL files.")
        splits = {split: load_jsonl(path) for split, path in expected_paths.items()}
    elif not FORCE_REDOWNLOAD and (legacy := load_local_ledgar_jsonl(LEGACY_LEDGAR_RAW_DIR)) is not None:
        print("Loading LEDGAR from existing legacy data/raw/lexglue_ledgar JSONL files.")
        splits = legacy
        for split, df in splits.items():
            save_jsonl(df, expected_paths[split])
    else:
        if not DOWNLOAD_LEDGAR_IF_MISSING:
            print("LEDGAR download is disabled and local JSONL files were not found.")
            return None
        try:
            print('Downloading LEDGAR via load_dataset("coastalcph/lex_glue", "ledgar").')
            dataset = load_dataset("coastalcph/lex_glue", "ledgar")
            splits = {}
            for split in ["train", "validation", "test"]:
                splits[split] = pd.DataFrame(dataset[split])
                save_jsonl(splits[split], expected_paths[split])
            label_feature = dataset["train"].features.get("label")
            if hasattr(label_feature, "names"):
                (LEDGAR_RAW_DIR / "label_names.txt").write_text("\n".join(label_feature.names) + "\n", encoding="utf-8")
        except Exception as exc:
            print(f"LEDGAR download failed: {type(exc).__name__}: {exc}")
            print("Place LEDGAR JSONL files in data/raw/ledgar/ as ledgar_train.jsonl, ledgar_validation.jsonl, and ledgar_test.jsonl.")
            return None

    for split, df in splits.items():
        print(f"LEDGAR {split}: {len(df)} rows, columns={list(df.columns)}")
    return splits


def copy_hf_file_to_raw(cache_path: str, target_name: str) -> Path:
    target = CUAD_RAW_DIR / target_name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(cache_path, target)
    return target


def download_cuad_if_missing() -> tuple[Path | None, Path | None]:
    """
    Download CUAD raw files from Hugging Face when missing.
    Returns paths to CUAD_v1.json and master_clauses.csv if found.
    """
    cuad_json_path = CUAD_RAW_DIR / "CUAD_v1.json"
    master_clauses_path = CUAD_RAW_DIR / "master_clauses.csv"

    if not FORCE_REDOWNLOAD and cuad_json_path.exists() and master_clauses_path.exists():
        print("Using existing CUAD files from data/raw/cuad/.")
        print(f"CUAD JSON: {cuad_json_path}")
        print(f"CUAD master clauses CSV: {master_clauses_path}")
        return cuad_json_path, master_clauses_path

    if not DOWNLOAD_CUAD_IF_MISSING:
        print("CUAD download is disabled.")
        return (cuad_json_path if cuad_json_path.exists() else None, master_clauses_path if master_clauses_path.exists() else None)

    try:
        print("Downloading CUAD_v1.json and master_clauses.csv with hf_hub_download.")
        json_cache_path = hf_hub_download(repo_id="theatticusproject/cuad", filename="CUAD_v1/CUAD_v1.json", repo_type="dataset")
        csv_cache_path = hf_hub_download(repo_id="theatticusproject/cuad", filename="CUAD_v1/master_clauses.csv", repo_type="dataset")
        cuad_json_path = copy_hf_file_to_raw(json_cache_path, "CUAD_v1.json")
        master_clauses_path = copy_hf_file_to_raw(csv_cache_path, "master_clauses.csv")
    except Exception as first_exc:
        print(f"Direct CUAD file download failed: {type(first_exc).__name__}: {first_exc}")
        try:
            print("Trying snapshot_download for the CUAD dataset repository.")
            snapshot_dir = snapshot_download(
                repo_id="theatticusproject/cuad",
                repo_type="dataset",
                local_dir=CUAD_RAW_DIR,
                local_dir_use_symlinks=False,
            )
            snapshot_dir = Path(snapshot_dir)
            json_candidates = list(snapshot_dir.rglob("CUAD_v1.json"))
            csv_candidates = list(snapshot_dir.rglob("master_clauses.csv"))
            if json_candidates:
                cuad_json_path = copy_hf_file_to_raw(str(json_candidates[0]), "CUAD_v1.json")
            if csv_candidates:
                master_clauses_path = copy_hf_file_to_raw(str(csv_candidates[0]), "master_clauses.csv")
        except Exception as second_exc:
            print(f"CUAD download failed or files were not found. CUAD extension will be skipped.")
            print(f"snapshot_download error: {type(second_exc).__name__}: {second_exc}")
            return None, None

    final_json = cuad_json_path if cuad_json_path.exists() else None
    final_csv = master_clauses_path if master_clauses_path.exists() else None
    print(f"CUAD JSON found: {final_json}")
    print(f"CUAD master clauses CSV found: {final_csv}")
    if final_json is None or final_csv is None:
        print("CUAD download failed or files were not found. CUAD extension will be skipped.")
    return final_json, final_csv


def load_cuad_raw_files(cuad_json_path: Path | None, master_clauses_path: Path | None) -> tuple[dict[str, Any] | None, pd.DataFrame | None]:
    """Load raw CUAD JSON and master_clauses.csv."""
    raw_cuad_json = None
    master_clauses_df = None
    if cuad_json_path and Path(cuad_json_path).exists():
        with Path(cuad_json_path).open("r", encoding="utf-8") as f:
            raw_cuad_json = json.load(f)
    if master_clauses_path and Path(master_clauses_path).exists():
        master_clauses_df = pd.read_csv(master_clauses_path)
    return raw_cuad_json, master_clauses_df


def normalise_whitespace(text: Any) -> str:
    if text is None:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()


def cuad_label_from_qa(qa: dict[str, Any]) -> str:
    for key in ["clause_type", "clause_category", "category", "label", "title"]:
        value = normalise_whitespace(qa.get(key))
        if value:
            return value
    qa_id = normalise_whitespace(qa.get("id"))
    if "__" in qa_id:
        return normalise_whitespace(qa_id.rsplit("__", 1)[-1].replace("_", " "))
    question = normalise_whitespace(qa.get("question"))
    quoted = re.search(r'related to ["“]([^"”]+)["”]', question, flags=re.IGNORECASE)
    if quoted:
        return normalise_whitespace(quoted.group(1))
    return question.strip(" ?.:")


def adapt_cuad_to_clause_classification(raw_cuad_json: dict[str, Any] | None) -> pd.DataFrame:
    """
    Convert CUAD from QA/span-extraction format into clause classification format.
    answer span text -> CUAD clause category
    """
    if raw_cuad_json is None:
        return pd.DataFrame(columns=["text", "label", "source_dataset", "source_id"])

    records = []
    for doc_idx, document in enumerate(raw_cuad_json.get("data", [])):
        source_id = normalise_whitespace(document.get("title") or document.get("id") or f"cuad_doc_{doc_idx}")
        for paragraph in document.get("paragraphs", []):
            for qa in paragraph.get("qas", []):
                if qa.get("is_impossible") is True:
                    continue
                label = cuad_label_from_qa(qa)
                if not label:
                    continue
                for answer in qa.get("answers", []) or []:
                    if isinstance(answer, dict):
                        span_text = normalise_whitespace(answer.get("text"))
                    else:
                        span_text = normalise_whitespace(answer)
                    if span_text:
                        records.append(
                            {
                                "text": span_text,
                                "label": label,
                                "source_dataset": "CUAD",
                                "source_id": source_id,
                            }
                        )
    df = pd.DataFrame(records)
    if df.empty:
        return pd.DataFrame(columns=["text", "label", "source_dataset", "source_id"])
    df = df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)
    return df[["text", "label", "source_dataset", "source_id"]]


ledgar_raw_splits = load_or_download_ledgar()
cuad_json_path, master_clauses_path = download_cuad_if_missing()
raw_cuad_json, master_clauses_df = load_cuad_raw_files(cuad_json_path, master_clauses_path)
cuad_clause_df = adapt_cuad_to_clause_classification(raw_cuad_json)

print("\nDataset availability:")
print(f"- LEDGAR downloaded/loaded: {'yes' if ledgar_raw_splits is not None else 'no'}")
if ledgar_raw_splits is not None:
    print(f"- LEDGAR train size: {len(ledgar_raw_splits['train'])}")
    print(f"- LEDGAR validation size: {len(ledgar_raw_splits['validation'])}")
    print(f"- LEDGAR test size: {len(ledgar_raw_splits['test'])}")
else:
    print("- LEDGAR train size: unavailable")
    print("- LEDGAR validation size: unavailable")
    print("- LEDGAR test size: unavailable")
print(f"- CUAD JSON found: {'yes' if cuad_json_path else 'no'}")
print(f"- CUAD master clauses CSV found: {'yes' if master_clauses_path else 'no'}")
if raw_cuad_json is not None:
    print(f"- CUAD adapted span examples available for optional analysis: {len(cuad_clause_df)}")

## 3. LEDGAR Preprocessing and EDA

The preprocessing standardises LEDGAR into a compact clause-classification schema, removes malformed examples and exact duplicate text-label pairs, selects the most frequent labels using only the training split, and preserves the official train/validation/test structure.

In [ ]:
REQUIRED_SCHEMA = ["text", "label", "label_id", "split", "source_dataset"]


def load_ledgar_label_names() -> list[str] | None:
    for path in [LEDGAR_RAW_DIR / "label_names.txt", PROJECT_ROOT / "outputs" / "label_names.txt", PROCESSED_DATA_DIR / "label_names.txt"]:
        if path.exists():
            labels = [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
            if labels:
                return labels
    return None


def detect_text_column(df: pd.DataFrame) -> str:
    for column in ["text", "provision", "clause", "contract_text"]:
        if column in df.columns:
            return column
    raise ValueError(f"Could not detect text column. Columns: {list(df.columns)}")


def detect_label_column(df: pd.DataFrame) -> str:
    for column in ["label", "labels", "category"]:
        if column in df.columns:
            return column
    raise ValueError(f"Could not detect label column. Columns: {list(df.columns)}")


def standardise_ledgar_split(df: pd.DataFrame, split: str, label_names: list[str] | None) -> pd.DataFrame:
    text_column = detect_text_column(df)
    label_column = detect_label_column(df)
    records = []
    for row in df.to_dict(orient="records"):
        text = normalise_whitespace(row.get(text_column))
        raw_label = row.get(label_column)
        if label_names is not None and isinstance(raw_label, (int, np.integer)) and 0 <= int(raw_label) < len(label_names):
            label = label_names[int(raw_label)]
        else:
            label = normalise_whitespace(raw_label)
        if text and label:
            records.append(
                {
                    "text": text,
                    "label": label,
                    "label_id": -1,
                    "split": split,
                    "source_dataset": "LEDGAR",
                }
            )
    standardised = pd.DataFrame(records, columns=REQUIRED_SCHEMA)
    return standardised.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)


def preprocess_ledgar(raw_splits: dict[str, pd.DataFrame] | None) -> tuple[dict[str, pd.DataFrame], dict[str, int], dict[int, str]]:
    if raw_splits is None:
        print("LEDGAR preprocessing skipped because the raw dataset is unavailable.")
        return {}, {}, {}

    label_names = load_ledgar_label_names()
    standardised = {split: standardise_ledgar_split(df, split, label_names) for split, df in raw_splits.items()}
    train_counts = standardised["train"]["label"].value_counts()
    selected_labels = train_counts.head(TOP_K_LABELS).index.tolist()
    label2id = {label: idx for idx, label in enumerate(selected_labels)}
    id2label = {idx: label for label, idx in label2id.items()}

    processed = {}
    for split, df in standardised.items():
        filtered = df[df["label"].isin(selected_labels)].copy().reset_index(drop=True)
        filtered["label_id"] = filtered["label"].map(label2id).astype(int)
        processed[split] = filtered[REQUIRED_SCHEMA]
        save_jsonl(processed[split], PROCESSED_DATA_DIR / f"ledgar_{split}.jsonl")

    (PROCESSED_DATA_DIR / "label_names.txt").write_text("\n".join(selected_labels) + "\n", encoding="utf-8")
    write_json(PROCESSED_DATA_DIR / "label_counts.json", train_counts.loc[selected_labels].astype(int).to_dict())
    summary = {
        "dataset": DATASET_NAME,
        "top_k_labels": TOP_K_LABELS,
        "rows_per_split": {split: int(len(df)) for split, df in processed.items()},
        "number_of_classes": len(selected_labels),
        "labels": selected_labels,
    }
    write_json(PROCESSED_DATA_DIR / "dataset_summary.json", summary)
    return processed, label2id, id2label


def create_ledgar_eda(processed: dict[str, pd.DataFrame]) -> pd.DataFrame:
    if not processed:
        return pd.DataFrame()
    eda_dir = RESULTS_DIR / "eda"
    eda_dir.mkdir(parents=True, exist_ok=True)
    combined = pd.concat(processed.values(), ignore_index=True)
    combined["word_count"] = combined["text"].str.split().str.len()
    label_counts = combined["label"].value_counts()

    fig, ax = plt.subplots(figsize=(12, 6))
    label_counts.plot(kind="bar", ax=ax)
    ax.set_title("LEDGAR Class Distribution")
    ax.set_xlabel("Label")
    ax.set_ylabel("Examples")
    ax.tick_params(axis="x", labelrotation=90, labelsize=7)
    fig.tight_layout()
    fig.savefig(eda_dir / "class_distribution.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 5))
    combined["word_count"].plot(kind="hist", bins=50, ax=ax)
    ax.set_title("LEDGAR Clause Length Histogram")
    ax.set_xlabel("Word count")
    ax.set_ylabel("Examples")
    fig.tight_layout()
    fig.savefig(eda_dir / "clause_length_histogram.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    examples = []
    for label in label_counts.index:
        for row in combined[combined["label"] == label].head(3).to_dict(orient="records"):
            examples.append({"label": label, "split": row["split"], "text": row["text"]})
    save_jsonl(pd.DataFrame(examples), eda_dir / "examples_per_label.jsonl")

    split_summary = pd.DataFrame(
        [{"split": split, "rows": len(df), "classes": df["label"].nunique()} for split, df in processed.items()]
    )
    split_summary.to_csv(eda_dir / "dataset_split_summary.csv", index=False)
    return split_summary


processed_splits, label2id, id2label = preprocess_ledgar(ledgar_raw_splits)
split_summary = create_ledgar_eda(processed_splits)
if processed_splits:
    train_df = processed_splits["train"]
    validation_df = processed_splits["validation"]
    test_df = processed_splits["test"]
    label_names = [id2label[i] for i in sorted(id2label)]
    display(split_summary)
    display(pd.DataFrame({"label": label_names}))
else:
    train_df = validation_df = test_df = pd.DataFrame(columns=REQUIRED_SCHEMA)
    label_names = []
    print("Main LEDGAR experiment cannot run without LEDGAR data.")

## 4. Shared Evaluation Utilities

All model families are evaluated with the same core metrics: accuracy, macro-F1, weighted-F1, a per-class classification report, and a confusion matrix. Macro-F1 is important because LEDGAR is class-imbalanced.

In [ ]:
completed_results: list[dict[str, Any]] = []
prediction_tables: dict[str, pd.DataFrame] = {}
trained_models: dict[str, Any] = {}


def safe_name(value: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", value.lower()).strip("_") or "item"


def plot_confusion(cm: np.ndarray, labels: list[str], output_path: Path, title: str) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    size = max(8, min(18, len(labels) * 0.55))
    fig, ax = plt.subplots(figsize=(size, size))
    image = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    positions = np.arange(len(labels))
    ax.set_xticks(positions)
    ax.set_yticks(positions)
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return output_path


def evaluate_predictions_common(
    model_family: str,
    model_name: str,
    training_type: str,
    y_true: list[int],
    y_pred: list[int],
    df: pd.DataFrame,
    output_dir: Path,
    notes: str = "",
    valid_labels: list[int] | None = None,
) -> dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)
    report_dir = output_dir / "classification_reports"
    cm_dir = output_dir / "confusion_matrices"
    labels = valid_labels or sorted(id2label)
    names = [id2label[i] for i in labels]
    report = classification_report(y_true, y_pred, labels=labels, target_names=names, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    report_path = write_json(report_dir / f"{safe_name(model_name)}_report.json", report)
    cm_path = plot_confusion(cm, names, cm_dir / f"{safe_name(model_name)}_confusion_matrix.png", f"{model_name} Confusion Matrix")
    pred_df = df[["text", "label", "label_id"]].copy()
    pred_df["predicted_label_id"] = y_pred
    pred_df["predicted_label"] = [id2label.get(int(pred), "INVALID_PREDICTION") for pred in y_pred]
    prediction_tables[model_name] = pred_df
    result = {
        "model_family": model_family,
        "model_name": model_name,
        "training_type": training_type,
        "dataset": DATASET_NAME,
        "eval_split": "test",
        "sample_size": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
        "notes": notes,
        "classification_report_path": str(report_path),
        "confusion_matrix_path": str(cm_path),
    }
    completed_results.append(result)
    return result

## 5. Dummy Baselines

Random and majority baselines establish lower-bound performance. They are useful checks before interpreting more complex supervised models.

In [ ]:
baseline_results = []

if train_df.empty:
    print("Baseline experiments skipped because LEDGAR data is unavailable.")
else:
    baseline_dir = RESULTS_DIR / "baselines"
    y_train = train_df["label_id"].astype(int).tolist()
    y_test = test_df["label_id"].astype(int).tolist()
    labels = sorted(id2label)
    rng = np.random.default_rng(SEED)

    uniform_pred = rng.choice(labels, size=len(test_df), replace=True).astype(int).tolist()
    baseline_results.append(
        evaluate_predictions_common(
            "baseline",
            "random_uniform",
            "dummy",
            y_test,
            uniform_pred,
            test_df,
            baseline_dir,
            "Uniform random over selected labels.",
        )
    )

    train_counts = train_df["label_id"].value_counts().sort_index()
    probabilities = np.array([train_counts.get(label, 0) for label in labels], dtype=float)
    probabilities = probabilities / probabilities.sum()
    distribution_pred = rng.choice(labels, size=len(test_df), replace=True, p=probabilities).astype(int).tolist()
    baseline_results.append(
        evaluate_predictions_common(
            "baseline",
            "random_train_distribution",
            "dummy",
            y_test,
            distribution_pred,
            test_df,
            baseline_dir,
            "Random predictions sampled from the training label distribution.",
        )
    )

    majority_label = int(train_df["label_id"].mode().iloc[0])
    majority_pred = [majority_label] * len(test_df)
    baseline_results.append(
        evaluate_predictions_common(
            "baseline",
            "majority_baseline",
            "dummy",
            y_test,
            majority_pred,
            test_df,
            baseline_dir,
            "Always predicts the most frequent training label.",
        )
    )

    baseline_results_df = pd.DataFrame(baseline_results)
    baseline_results_df.to_csv(baseline_dir / "baseline_results.csv", index=False)
    display(baseline_results_df[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

## 6. Classical TF-IDF Models

TF-IDF Logistic Regression, Linear SVM, and optional Multinomial Naive Bayes provide efficient sparse-text baselines. The validation split is used for model selection, then the selected configuration is evaluated once on the test split.

In [ ]:
classical_results = []
best_classical_model = None
best_classical_name = None
best_classical_validation_macro_f1 = -1.0


def classical_model_configs() -> list[dict[str, Any]]:
    configs = []
    for max_features in MAX_FEATURES_LIST:
        for ngram_range in NGRAM_RANGES:
            configs.append({"max_features": max_features, "ngram_range": ngram_range})
    return configs


def build_classical_pipeline(model_name: str, max_features: int, ngram_range: tuple[int, int], class_weight: str | None = None) -> Pipeline:
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range, lowercase=True, stop_words=None)
    if model_name == "logistic_regression":
        classifier = LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=SEED, n_jobs=-1)
    elif model_name == "linear_svm":
        classifier = LinearSVC(class_weight=class_weight, random_state=SEED)
    elif model_name == "multinomial_nb":
        classifier = MultinomialNB()
    else:
        raise ValueError(model_name)
    return Pipeline([("vectorizer", vectorizer), ("classifier", classifier)])


if not RUN_CLASSICAL_MODELS:
    print("Classical models skipped because RUN_CLASSICAL_MODELS is False.")
elif train_df.empty:
    print("Classical models skipped because LEDGAR data is unavailable.")
else:
    classical_dir = RESULTS_DIR / "classical"
    classical_dir.mkdir(parents=True, exist_ok=True)
    x_train = train_df["text"].tolist()
    y_train = train_df["label_id"].astype(int).tolist()
    x_val = validation_df["text"].tolist()
    y_val = validation_df["label_id"].astype(int).tolist()
    x_test = test_df["text"].tolist()
    y_test = test_df["label_id"].astype(int).tolist()

    model_names = ["logistic_regression", "linear_svm"]
    if RUN_NAIVE_BAYES:
        model_names.append("multinomial_nb")

    validation_rows = []
    for model_name in model_names:
        class_weight_options = [None]
        if model_name in {"logistic_regression", "linear_svm"}:
            class_weight_options.append("balanced")

        best_for_model = None
        best_for_model_score = -1.0
        best_for_model_config = None
        for config in classical_model_configs():
            for class_weight in class_weight_options:
                pipeline = build_classical_pipeline(model_name, class_weight=class_weight, **config)
                pipeline.fit(x_train, y_train)
                val_pred = pipeline.predict(x_val).astype(int).tolist()
                macro_f1 = f1_score(y_val, val_pred, labels=sorted(id2label), average="macro", zero_division=0)
                row = {
                    "model_name": model_name,
                    "validation_macro_f1": macro_f1,
                    "max_features": config["max_features"],
                    "ngram_range": str(config["ngram_range"]),
                    "class_weight": class_weight,
                }
                validation_rows.append(row)
                if macro_f1 > best_for_model_score:
                    best_for_model = pipeline
                    best_for_model_score = macro_f1
                    best_for_model_config = row

        test_pred = best_for_model.predict(x_test).astype(int).tolist()
        notes = f"Selected on validation macro-F1. Config={best_for_model_config}"
        result = evaluate_predictions_common("classical", model_name, "supervised", y_test, test_pred, test_df, classical_dir, notes)
        result.update(best_for_model_config)
        classical_results.append(result)

        if best_for_model_score > best_classical_validation_macro_f1:
            best_classical_model = best_for_model
            best_classical_name = model_name
            best_classical_validation_macro_f1 = best_for_model_score

    classical_results_df = pd.DataFrame(classical_results)
    validation_grid_df = pd.DataFrame(validation_rows)
    classical_results_df.to_csv(classical_dir / "classical_results.csv", index=False)
    validation_grid_df.to_csv(classical_dir / "classical_validation_grid.csv", index=False)
    if best_classical_model is not None:
        joblib.dump(best_classical_model, classical_dir / "best_classical_model.pkl")
        joblib.dump(best_classical_model.named_steps["vectorizer"], classical_dir / "vectorizer.pkl")
        trained_models["best_classical"] = best_classical_model

    fig, ax = plt.subplots(figsize=(9, 5))
    classical_results_df.plot(kind="bar", x="model_name", y="macro_f1", ax=ax, legend=False)
    ax.set_title("Classical Model Test Macro-F1")
    ax.set_xlabel("Model")
    ax.set_ylabel("Macro-F1")
    fig.tight_layout()
    fig.savefig(classical_dir / "classical_model_comparison.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    display(classical_results_df[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

## 7. Fine-Tuned Transformer Classifier

DistilBERT is used as the preferred transformer because it is faster and more practical for coursework hardware. LegalBERT can be substituted through the configuration if GPU resources allow. This section catches memory or environment failures and records the skip instead of stopping the notebook.

In [ ]:
transformer_result = None
transformer_predictions_df = pd.DataFrame()


def transformer_compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, labels=sorted(id2label), average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, predictions, labels=sorted(id2label), average="weighted", zero_division=0),
    }


def training_arguments_kwargs(TrainingArguments, kwargs: dict[str, Any]) -> dict[str, Any]:
    import inspect

    params = inspect.signature(TrainingArguments.__init__).parameters
    adapted = dict(kwargs)
    if "eval_strategy" in params and "evaluation_strategy" in adapted:
        adapted["eval_strategy"] = adapted.pop("evaluation_strategy")
    return {key: value for key, value in adapted.items() if key in params}


if not RUN_TRANSFORMER:
    print("Transformer section skipped because RUN_TRANSFORMER is False.")
elif train_df.empty:
    print("Transformer section skipped because LEDGAR data is unavailable.")
elif not torch.cuda.is_available():
    print("Transformer training skipped because GPU/CUDA is unavailable in this runtime.")
else:
    try:
        ensure_package("transformers", "transformers")
        ensure_package("accelerate", "accelerate")
        from datasets import Dataset
        from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
        try:
            from transformers import EarlyStoppingCallback
            callbacks = [EarlyStoppingCallback(early_stopping_patience=1)]
        except Exception:
            callbacks = []

        transformer_dir = RESULTS_DIR / "transformer"
        transformer_dir.mkdir(parents=True, exist_ok=True)
        tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL_NAME)

        def to_hf_dataset(df: pd.DataFrame):
            data = df[["text", "label_id"]].rename(columns={"label_id": "labels"}).copy()
            data["labels"] = data["labels"].astype(int)
            dataset = Dataset.from_pandas(data, preserve_index=False)
            return dataset.map(
                lambda batch: tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_TRANSFORMER_LENGTH),
                batched=True,
                remove_columns=["text"],
            )

        train_dataset = to_hf_dataset(train_df)
        val_dataset = to_hf_dataset(validation_df)
        test_dataset = to_hf_dataset(test_df)

        model = AutoModelForSequenceClassification.from_pretrained(
            TRANSFORMER_MODEL_NAME,
            num_labels=len(id2label),
            id2label={int(k): v for k, v in id2label.items()},
            label2id={v: int(k) for k, v in id2label.items()},
        )

        batch_size = 16 if torch.cuda.get_device_properties(0).total_memory > 12_000_000_000 else 8
        args_dict = {
            "output_dir": str(transformer_dir / "checkpoints"),
            "evaluation_strategy": "epoch",
            "save_strategy": "epoch",
            "learning_rate": 2e-5,
            "per_device_train_batch_size": batch_size,
            "per_device_eval_batch_size": batch_size,
            "num_train_epochs": 3,
            "weight_decay": 0.01,
            "load_best_model_at_end": True,
            "metric_for_best_model": "macro_f1",
            "greater_is_better": True,
            "save_total_limit": 1,
            "report_to": [],
            "fp16": torch.cuda.is_available(),
            "seed": SEED,
        }
        training_args = TrainingArguments(**training_arguments_kwargs(TrainingArguments, args_dict))
        write_json(transformer_dir / "training_args.json", args_dict)

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=tokenizer,
            compute_metrics=transformer_compute_metrics,
            callbacks=callbacks,
        )
        trainer.train()
        output = trainer.predict(test_dataset)
        transformer_pred = np.argmax(output.predictions, axis=-1).astype(int).tolist()
        y_test = test_df["label_id"].astype(int).tolist()
        transformer_result = evaluate_predictions_common(
            "transformer",
            TRANSFORMER_MODEL_NAME,
            "fine-tuned supervised",
            y_test,
            transformer_pred,
            test_df,
            transformer_dir,
            "Fine-tuned Hugging Face sequence classifier.",
        )
        pd.DataFrame([transformer_result]).to_csv(transformer_dir / "transformer_results.csv", index=False)
        shutil.copy2(transformer_result["classification_report_path"], transformer_dir / "classification_report.json")
        shutil.copy2(transformer_result["confusion_matrix_path"], transformer_dir / "confusion_matrix.png")
        trainer.save_model(transformer_dir / "model")
        tokenizer.save_pretrained(transformer_dir / "model")
        trained_models["transformer_trainer"] = trainer
        transformer_predictions_df = prediction_tables[TRANSFORMER_MODEL_NAME]
        display(pd.DataFrame([transformer_result])[["model_name", "accuracy", "macro_f1", "weighted_f1"]])
    except Exception as exc:
        print(f"Transformer training failed or was skipped: {type(exc).__name__}: {exc}")
        completed_results.append(
            {
                "model_family": "transformer",
                "model_name": TRANSFORMER_MODEL_NAME,
                "training_type": "fine-tuned supervised",
                "dataset": DATASET_NAME,
                "eval_split": "test",
                "sample_size": 0,
                "accuracy": np.nan,
                "macro_f1": np.nan,
                "weighted_f1": np.nan,
                "notes": f"Skipped/failed: {type(exc).__name__}: {exc}",
            }
        )

## 8. Qwen2.5-Instruct Prompting Baseline

Qwen2.5-Instruct is used only as a zero-shot and few-shot prompting baseline. It is not fine-tuned. This tests whether an instruction-tuned causal language model can map legal clauses to the allowed LEDGAR labels without task-specific training.

In [ ]:
qwen_results_df = pd.DataFrame()
qwen_predictions_df = pd.DataFrame()
qwen_invalid_outputs_df = pd.DataFrame()
qwen_model = None
qwen_tokenizer = None


def build_allowed_labels_text() -> str:
    return "\n".join(f"- {label}" for label in label_names)


def parse_llm_label(output: str, labels: list[str]) -> str:
    cleaned = normalise_whitespace(output).splitlines()[0].strip(" `\"'") if output else ""
    if cleaned in labels:
        return cleaned
    lowered = {label.lower(): label for label in labels}
    if cleaned.lower() in lowered:
        return lowered[cleaned.lower()]
    containing = [label for label in labels if label.lower() in cleaned.lower()]
    if len(containing) == 1:
        return containing[0]
    matches = difflib.get_close_matches(cleaned, labels, n=2, cutoff=0.80)
    if len(matches) == 1:
        return matches[0]
    return "INVALID_PREDICTION"


def make_zero_shot_prompt(clause_text: str) -> str:
    return f"""You are a legal NLP classifier. Classify the following contract clause into exactly one of the allowed labels. Return only the label name.

Allowed labels:
{build_allowed_labels_text()}

Clause:
{clause_text}

Predicted label:"""


def build_few_shot_examples() -> list[dict[str, str]]:
    examples = []
    for label in label_names:
        label_rows = train_df[train_df["label"] == label].head(QWEN_FEW_SHOT_EXAMPLES_PER_CLASS)
        for row in label_rows.to_dict(orient="records"):
            examples.append({"text": row["text"], "label": row["label"]})
    return examples


few_shot_examples = build_few_shot_examples() if not train_df.empty else []


def make_few_shot_prompt(clause_text: str) -> str:
    demonstrations = []
    for example in few_shot_examples:
        demonstrations.append(f"Clause: {example['text']}\nPredicted label: {example['label']}")
    demo_text = "\n\n".join(demonstrations)
    return f"""You are a legal NLP classifier. Classify the following contract clause into exactly one of the allowed labels. Return only the label name.

Allowed labels:
{build_allowed_labels_text()}

Examples:
{demo_text}

Clause:
{clause_text}

Predicted label:"""


def qwen_generate_label(prompt: str) -> tuple[str, str]:
    inputs = qwen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(qwen_model.device)
    with torch.no_grad():
        output_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=24,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    generated = qwen_tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated, parse_llm_label(generated, label_names)


def evaluate_qwen_predictions(mode: str, rows: list[dict[str, Any]], output_dir: Path) -> dict[str, Any]:
    df = pd.DataFrame(rows)
    valid_mask = df["predicted_label"] != "INVALID_PREDICTION"
    invalid_rate = 1.0 - float(valid_mask.mean()) if len(df) else 1.0
    fallback_label_id = int(train_df["label_id"].mode().iloc[0])
    y_true = df["label_id"].astype(int).tolist()
    y_pred = [
        int(label2id[pred]) if pred in label2id else fallback_label_id
        for pred in df["predicted_label"].tolist()
    ]
    result = evaluate_predictions_common(
        "llm_prompting",
        f"qwen_{mode}",
        mode,
        y_true,
        y_pred,
        df.rename(columns={"predicted_label": "model_output_label"}),
        output_dir,
        f"Qwen prompting baseline. Invalid prediction rate={invalid_rate:.4f}",
    )
    result["invalid_prediction_rate"] = invalid_rate
    return result


if not RUN_QWEN_BASELINE:
    print("Qwen baseline skipped because RUN_QWEN_BASELINE is False.")
elif train_df.empty:
    print("Qwen baseline skipped because LEDGAR data is unavailable.")
elif not torch.cuda.is_available():
    print("Qwen baseline skipped because GPU/CUDA is unavailable. Loading a 3B model on CPU is not practical for this notebook.")
else:
    try:
        ensure_package("transformers", "transformers")
        ensure_package("accelerate", "accelerate")
        from transformers import AutoModelForCausalLM, AutoTokenizer

        qwen_dir = RESULTS_DIR / "qwen"
        qwen_dir.mkdir(parents=True, exist_ok=True)
        qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
        qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
        qwen_model.eval()

        sample_size = min(QWEN_EVAL_SAMPLE_SIZE, len(test_df))
        qwen_sample = test_df.sample(n=sample_size, random_state=SEED).reset_index(drop=True)
        all_rows = []
        prompt_examples_text = []
        for mode, prompt_builder in [("zero_shot", make_zero_shot_prompt), ("few_shot", make_few_shot_prompt)]:
            mode_rows = []
            for row in qwen_sample.to_dict(orient="records"):
                prompt = prompt_builder(row["text"])
                raw_output, parsed_label = qwen_generate_label(prompt)
                mode_rows.append(
                    {
                        "mode": mode,
                        "text": row["text"],
                        "label": row["label"],
                        "label_id": int(row["label_id"]),
                        "raw_output": raw_output,
                        "predicted_label": parsed_label,
                    }
                )
            all_rows.extend(mode_rows)
            qwen_result = evaluate_qwen_predictions(mode, mode_rows, qwen_dir)
            prompt_examples_text.append(f"--- {mode} prompt example ---\n{prompt_builder(qwen_sample.iloc[0]['text'])}\n")

        qwen_predictions_df = pd.DataFrame(all_rows)
        qwen_predictions_df.to_csv(qwen_dir / "qwen_predictions.csv", index=False)
        qwen_results_df = pd.DataFrame([row for row in completed_results if str(row.get("model_name", "")).startswith("qwen_")])
        qwen_results_df.to_csv(qwen_dir / "qwen_results.csv", index=False)
        (qwen_dir / "qwen_prompt_examples.txt").write_text("\n\n".join(prompt_examples_text), encoding="utf-8")
        qwen_invalid_outputs_df = qwen_predictions_df[qwen_predictions_df["predicted_label"] == "INVALID_PREDICTION"].copy()
        qwen_invalid_outputs_df.to_csv(qwen_dir / "qwen_invalid_outputs.csv", index=False)
        display(qwen_results_df[["model_name", "accuracy", "macro_f1", "weighted_f1", "invalid_prediction_rate"]])
    except Exception as exc:
        print(f"Qwen baseline could not load or run: {type(exc).__name__}: {exc}")
        completed_results.append(
            {
                "model_family": "llm_prompting",
                "model_name": "qwen_skipped",
                "training_type": "zero/few-shot prompting",
                "dataset": DATASET_NAME,
                "eval_split": "test",
                "sample_size": 0,
                "accuracy": np.nan,
                "macro_f1": np.nan,
                "weighted_f1": np.nan,
                "notes": f"Skipped/failed: {type(exc).__name__}: {exc}",
            }
        )

## 9. Small Agentic Review Prototype

This section is a small human-in-the-loop demonstration inspired by ReAct/tool-use workflows. It is not a fully autonomous legal agent. It uses a supervised classifier for clause triage, flags low-confidence cases for human review, and optionally asks Qwen for a short research-oriented explanation. This output is for clause triage and research purposes only.

In [ ]:
agentic_examples_df = pd.DataFrame()


def classifier_confidence(model: Pipeline, texts: list[str]) -> tuple[list[int], list[float]]:
    predictions = model.predict(texts).astype(int).tolist()
    classifier = model.named_steps.get("classifier")
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(texts)
        confidence = probabilities.max(axis=1).astype(float).tolist()
    elif hasattr(classifier, "decision_function"):
        margins = classifier.decision_function(model.named_steps["vectorizer"].transform(texts))
        margins = np.atleast_2d(margins)
        sorted_scores = np.sort(margins, axis=1)
        if sorted_scores.shape[1] > 1:
            raw_margin = sorted_scores[:, -1] - sorted_scores[:, -2]
        else:
            raw_margin = np.abs(sorted_scores[:, -1])
        confidence = (raw_margin / (1.0 + raw_margin)).astype(float).tolist()
    else:
        confidence = [np.nan] * len(texts)
    return predictions, confidence


def qwen_triage_explanation(clause_text: str, predicted_label: str, confidence: float) -> str:
    if qwen_model is None or qwen_tokenizer is None:
        return "Qwen explanation unavailable in this run."
    prompt = f"""This output is for clause triage and research purposes only. It is not legal advice.
Predicted clause type: {predicted_label}
Classifier confidence: {confidence:.3f}

Clause:
{clause_text}

Write one short explanation with: predicted clause type, supporting phrase, uncertainty note, and no legal advice."""
    inputs = qwen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536).to(qwen_model.device)
    with torch.no_grad():
        output_ids = qwen_model.generate(**inputs, max_new_tokens=96, do_sample=False, pad_token_id=qwen_tokenizer.eos_token_id)
    return qwen_tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


if not RUN_AGENTIC_EXTENSION:
    print("Agentic review prototype skipped because RUN_AGENTIC_EXTENSION is False.")
elif best_classical_model is None:
    print("Agentic review prototype skipped because no supervised classifier is available.")
else:
    review_dir = RESULTS_DIR / "agentic_review"
    review_dir.mkdir(parents=True, exist_ok=True)
    sample = test_df.sample(n=min(20, len(test_df)), random_state=SEED).reset_index(drop=True)
    pred_ids, confidences = classifier_confidence(best_classical_model, sample["text"].tolist())
    rows = []
    threshold = 0.55
    for row, pred_id, confidence in zip(sample.to_dict(orient="records"), pred_ids, confidences):
        predicted_label = id2label[int(pred_id)]
        requires_review = bool(np.isnan(confidence) or confidence < threshold)
        explanation = ""
        if requires_review:
            explanation = qwen_triage_explanation(row["text"], predicted_label, confidence if not np.isnan(confidence) else 0.0)
        rows.append(
            {
                "text": row["text"],
                "true_label": row["label"],
                "predicted_label": predicted_label,
                "confidence": confidence,
                "requires_human_review": requires_review,
                "triage_note": "This output is for clause triage and research purposes only.",
                "optional_qwen_explanation": explanation,
            }
        )
    agentic_examples_df = pd.DataFrame(rows)
    agentic_examples_df.to_csv(review_dir / "agentic_examples.csv", index=False)
    display(agentic_examples_df.head(10))

## 10. Final Model Comparison

The final table records every completed or skipped model family without fabricating missing metrics. Prompted LLM results are based on the configured test sample size rather than the full test set.

In [ ]:
comparison_df = pd.DataFrame(completed_results)
comparison_columns = [
    "model_family",
    "model_name",
    "training_type",
    "dataset",
    "eval_split",
    "sample_size",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "notes",
]
comparison_df = comparison_df.reindex(columns=comparison_columns)
comparison_path = RESULTS_DIR / "final_model_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

plot_df = comparison_df.dropna(subset=["macro_f1"]).copy()
if not plot_df.empty:
    fig, ax = plt.subplots(figsize=(11, 5))
    plot_df.sort_values("macro_f1", ascending=False).plot(kind="bar", x="model_name", y="macro_f1", ax=ax, legend=False)
    ax.set_title("Final Model Comparison by Macro-F1")
    ax.set_xlabel("Model")
    ax.set_ylabel("Macro-F1")
    ax.tick_params(axis="x", labelrotation=45)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "final_model_comparison.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Saved final comparison to: {comparison_path}")
display(comparison_df)

## 11. Error Analysis

This section examines typical error patterns: confused label pairs, misclassified examples, class imbalance, label ambiguity, long clauses, and boilerplate wording. The discussion should be based on the observed outputs from the current run, not invented results.

In [ ]:
error_dir = RESULTS_DIR / "error_analysis"
error_dir.mkdir(parents=True, exist_ok=True)


def top_confusions(pred_df: pd.DataFrame, top_n: int = 15) -> pd.DataFrame:
    errors = pred_df[pred_df["label_id"] != pred_df["predicted_label_id"]].copy()
    if errors.empty:
        return pd.DataFrame(columns=["label", "predicted_label", "count"])
    return (
        errors.groupby(["label", "predicted_label"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(top_n)
    )


best_completed = comparison_df.dropna(subset=["macro_f1"]).sort_values("macro_f1", ascending=False)
best_model_name = best_completed.iloc[0]["model_name"] if not best_completed.empty else None
print(f"Best completed model by macro-F1: {best_model_name}")

if best_classical_name and best_classical_name in prediction_tables:
    best_classical_errors = prediction_tables[best_classical_name]
else:
    classical_available = [name for name in ["logistic_regression", "linear_svm", "multinomial_nb"] if name in prediction_tables]
    best_classical_errors = prediction_tables[classical_available[0]] if classical_available else pd.DataFrame()

if not best_classical_errors.empty:
    classical_confusions = top_confusions(best_classical_errors)
    classical_misclassified = best_classical_errors[best_classical_errors["label_id"] != best_classical_errors["predicted_label_id"]].head(10)
    classical_confusions.to_csv(error_dir / "classical_top_confusions.csv", index=False)
    classical_misclassified.to_csv(error_dir / "classical_misclassified_examples.csv", index=False)
    print("Top classical confused label pairs:")
    display(classical_confusions)
    print("Classical misclassified examples:")
    display(classical_misclassified[["text", "label", "predicted_label"]])

if TRANSFORMER_MODEL_NAME in prediction_tables:
    transformer_errors = prediction_tables[TRANSFORMER_MODEL_NAME]
    transformer_confusions = top_confusions(transformer_errors)
    transformer_misclassified = transformer_errors[transformer_errors["label_id"] != transformer_errors["predicted_label_id"]].head(10)
    transformer_confusions.to_csv(error_dir / "transformer_top_confusions.csv", index=False)
    transformer_misclassified.to_csv(error_dir / "transformer_misclassified_examples.csv", index=False)
    print("Transformer misclassified examples:")
    display(transformer_misclassified[["text", "label", "predicted_label"]])

    if not best_classical_errors.empty:
        comparison_errors = pd.DataFrame(
            {
                "text": best_classical_errors["text"],
                "true_label": best_classical_errors["label"],
                "classical_predicted": best_classical_errors["predicted_label"],
                "transformer_predicted": transformer_errors["predicted_label"],
            }
        )
        comparison_errors["classical_correct"] = comparison_errors["true_label"] == comparison_errors["classical_predicted"]
        comparison_errors["transformer_correct"] = comparison_errors["true_label"] == comparison_errors["transformer_predicted"]
        comparison_errors.to_csv(error_dir / "classical_vs_transformer_errors.csv", index=False)
        display(comparison_errors.head(10))
else:
    print("Transformer error analysis skipped because transformer predictions are unavailable.")

if not qwen_invalid_outputs_df.empty:
    print("Qwen invalid outputs:")
    display(qwen_invalid_outputs_df.head(10))
else:
    print("No Qwen invalid outputs available, or Qwen did not run.")

if not qwen_predictions_df.empty:
    plausible_nonmatching = qwen_predictions_df[
        (qwen_predictions_df["predicted_label"] != "INVALID_PREDICTION")
        & (qwen_predictions_df["predicted_label"] != qwen_predictions_df["label"])
    ].head(10)
    plausible_nonmatching.to_csv(error_dir / "qwen_plausible_nonmatching_examples.csv", index=False)
    print("Qwen semantically plausible but non-matching label examples require manual inspection:")
    display(plausible_nonmatching[["mode", "text", "label", "raw_output", "predicted_label"]])

imbalance = train_df["label"].value_counts().rename_axis("label").reset_index(name="train_count") if not train_df.empty else pd.DataFrame()
imbalance.to_csv(error_dir / "class_imbalance.csv", index=False)
print("Class imbalance summary:")
display(imbalance.head(20))

print("Interpretive prompts for write-up:")
print("- Compare macro-F1 with weighted-F1 to assess class imbalance effects.")
print("- Inspect confused label pairs for annotation ambiguity and overlapping boilerplate wording.")
print("- Review long misclassified clauses; truncation and mixed clause content may affect models.")
print("- For Qwen, compare invalid outputs and zero-shot vs few-shot changes without treating prompted LLMs as directly equivalent to fine-tuned classifiers.")

## 12. Report-Ready Method Notes

**Aim.** Compare classical supervised models, fine-tuned transformer models, and instruction-tuned LLM prompting for LEDGAR legal clause classification.

**Dataset.** LEDGAR provides legal clause/provision texts with clause type labels and official train/validation/test splits. CUAD is downloaded separately because it has a span-extraction contract review format and is not merged into LEDGAR.

**Preprocessing.** The pipeline standardises columns, normalises whitespace only, removes empty examples and exact duplicate text-label pairs, selects the top-k labels using the training split, and preserves official splits.

**Baselines.** Random and majority baselines establish lower-bound performance for the selected label set.

**Classical models.** TF-IDF Logistic Regression and Linear SVM are efficient sparse-text baselines; Naive Bayes is included as an optional comparison.

**Transformer.** DistilBERT or LegalBERT tests whether contextual representations improve clause classification when fine-tuned on LEDGAR.

**Qwen.** Qwen2.5-Instruct is used as a zero-shot/few-shot prompting baseline to test whether an instruction-tuned LLM can classify clauses without task-specific fine-tuning.

**Agentic extension.** The prototype demonstrates a human-in-the-loop clause triage workflow using classifier confidence and optional LLM explanation. This output is for clause triage and research purposes only.

**Limitations.** LEDGAR labels are clause types, not legal risk labels. The models do not provide legal advice. Class imbalance affects macro-F1. Qwen prompting may be sensitive to prompt format. Comparing fine-tuned classifiers and prompted LLMs is not perfectly fair because their training/setup differs. The agentic workflow is illustrative and not production-ready.